# Getting started with WFDB-x

`wfdbx` reads and writes physiological signal records in the WFDB format
used by [PhysioNet](https://physionet.org). This notebook reads the header
and the signals of a record.

To install the package with matplotlib and Jupyter, and copy these notebooks
to a directory, run in a shell:

```
pip install "wfdb-x[examples]"
wfdbx --examples my-examples
```

In [ ]:
import numpy as np
import wfdbx

# The sample record 100s, included with WFDB-x, is the first minute of
# record 100 of the MIT-BIH Arrhythmia Database. Read it from the
# package's data directory, wfdbx.sample_data_dir().
wfdbx.wfdbx_set_input_dir(wfdbx.sample_data_dir())

print("WFDB-x", wfdbx.__version__, "- WFDB", wfdbx.wfdbversion())
print("input directory:", wfdbx.wfdbx_get_input_dir())

## Where records are found

A record is a set of files with a common name: a header (`100s.hea`), one
or more signal files (`100s.dat`) and annotation files (`100s.atr`).

A record name such as `100s` is read from the **input directory**. The cell
above sets it to the package's data directory with `wfdbx_set_input_dir`.
By default, it is the current directory. A full path, a `~` path or a URL is
used as given. Files you write go to the **output directory**, which you set
with `wfdbx_set_output_dir`.

The sample record `100s` is the first minute of record 100 of the
[MIT-BIH Arrhythmia Database](https://physionet.org/content/mitdb/): two ECG
signals sampled at 360 Hz.

## The header

`read_header` returns a `Header`. It holds the sampling frequency, the
number of samples and of signals, the start time and date, the comments,
and one `Siginfo` object per signal in `siginfo`.

In [ ]:
hdr = wfdbx.read_header("100s")
print(hdr)
for s in hdr.siginfo:
    print(f"{s.desc:5s} format {s.fmt}, gain {s.gain:g} adu per unit, "
          f"{s.adcres}-bit ADC, baseline {s.baseline}, units {s.units!r}")

The header of this record gives no physical units, so `units` is `None`.
wfdb-x reports a value that a record does not give as missing. PhysioNet
documents the units of this database as millivolts.

## The signals

`read_record` reads a record, or part of it, into a NumPy array with one
column per signal. With `physical=True`, the samples are converted to
physical units with the gains and baselines of the header.

In [ ]:
rec = wfdbx.read_record("100s", physical=True)
print(rec)
print("sampling frequency:", rec.fs, "Hz")
print("signals:", rec.names, "units:", rec.units, "gains:", rec.gains)
print("duration:", len(rec.signals) / rec.fs, "s")

Times can be sample numbers or WFDB time strings: `"0:10"` is 10 seconds
and `"s3600"` is sample 3600. You can select signals by number or by name.

In [ ]:
part = wfdbx.read_record("100s", channels=["MLII", "V5"],
                         sampfrom="0:10", sampto="0:13", physical=True)
t = (part.sampfrom + np.arange(len(part.signals))) / part.fs
print(part.signals.shape, "samples from", t[0], "s to", t[-1], "s")

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:  # the plots are optional
    plt = None
    print("matplotlib is not installed: pip install matplotlib "
          "(or pip install \"wfdb-x[examples]\")")

if plt:
    fig, axes = plt.subplots(len(part.names), 1, sharex=True, figsize=(9, 4))
    for k, ax in enumerate(axes):
        ax.plot(t, part.signals[:, k], linewidth=0.8)
        ax.set_ylabel(f"{part.names[k]}\n({part.units[k] or 'units not given'})")
    axes[-1].set_xlabel("time (s)")
    plt.tight_layout()
    plt.show()

## ADC units and physical units

Without `physical=True`, the samples are the integers stored in the signal
file (ADC units). The conversion to physical units is
`(sample - baseline) / gain`:

In [ ]:
raw = wfdbx.read_record("100s", sampto=5)
print(raw.signals)
print((raw.signals - raw.baselines) / raw.gains)